In [0]:
print("Aegis Fraud Detection - Project Started")
print(spark.version)

Aegis Fraud Detection - Project Started
4.1.0


## PHASE 1 : DATA EXPLORATION

Step 1 — Loading the CSV and looking at its shape

In [0]:
# Loading the PaySim dataset from where you uploaded it
df = spark.read.csv(
    "dbfs:/Volumes/workspace/default/raw_data/",
    header=True,       # First row contains column names
    inferSchema=True   # Spark will guess the data type of each column
)

# Printing the shape: how many rows and columns do we have?
print("Row count:", df.count())
print("Column count:", len(df.columns))

Row count: 50500
Column count: 11


Step 2 — See what the columns are named and what types Spark gave them

In [0]:
# Showing the column names and the data type Spark assigned to each one
df.printSchema()

root
 |-- step: double (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: double (nullable = true)
 |-- isFlaggedFraud: double (nullable = true)



Step 3 — Checking for the amount problem and looking at real rows

In [0]:
# Showing the first 5 rows so we can see real values
# .show() prints a table directly in the notebook
df.show(5, truncate=False)

+-----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step |type    |amount   |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+-----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|396.0|CASH_OUT|118326.41|C944557061 |121796.0     |3469.59       |C1487077526|1182256.11    |1300582.52    |0.0    |0.0           |
|373.0|PAYMENT |17068.83 |C255803662 |20451.0      |3382.17       |M499251811 |0.0           |0.0           |0.0    |0.0           |
|133.0|CASH_OUT|369264.57|C1339328459|124268.0     |0.0           |C1297375011|868843.17     |1238107.74    |0.0    |0.0           |
|351.0|PAYMENT |10039.08 |C985426754 |0.0          |0.0           |M1813471006|0.0           |0.0           |0.0    |0.0           |
|257.0|PAYMENT |7639.89  |C1374724372|0.0          |0.0           |M1

Step 4 — Checking class imbalance (the most important number in this project)

In [0]:
from pyspark.sql.functions import col

# Counting how many transactions are fraud vs legitimate
fraud_counts = df.groupBy("isFraud").count()
fraud_counts.show()

+--------------------+-----+
|             isFraud|count|
+--------------------+-----+
|0.002859836613431...|    1|
|-0.00879539391254...|    1|
|-6.01750419846282...|    1|
|                 1.0|   70|
|-0.00463208923901...|    1|
| -0.0055726559985202|    1|
|-0.00239237942144...|    1|
|0.002133231409595253|    1|
|                 0.0|50397|
|-0.00246560258354...|    1|
|0.006180874392786024|    1|
|-0.00713764559742...|    1|
|-0.00568097296162...|    1|
|-0.00199100367367...|    1|
|-0.00548625725869...|    1|
|0.003854091971737...|    1|
|1.648178318578788...|    1|
|0.003363769408909872|    1|
|-0.00682507584036...|    1|
|0.002661534486020309|    1|
+--------------------+-----+
only showing top 20 rows


In [0]:
# Calculating what percentage of transactions are fraud
total = df.count()
fraud_total = df.filter(col("isFraud") == 1).count()

print(f"Total transactions: {total}")
print(f"Fraudulent transactions: {fraud_total}")
print(f"Fraud rate: {round(fraud_total / total * 100, 3)}%")

Total transactions: 50500
Fraudulent transactions: 70
Fraud rate: 0.139%


There is a data quality problem that is extremely common in real-world datasets. The most likely cause: CSV file contains some rows where the columns shifted — meaning a value that belongs in one column ended up in a neighboring column. PaySim is a synthetic dataset generated by a simulator, and occasionally the simulation outputs malformed rows. There are garbage values that somehow ended up in the isFraud column.

Step 5 — Understanding the damage before fixing it

In [0]:
# Counting rows where isFraud is NOT 0 and NOT 1
# These are the corrupted rows
bad_rows = df.filter(
    (col("isFraud") != 0.0) & (col("isFraud") != 1.0)
).count()

print(f"Corrupted rows: {bad_rows}")
print(f"Clean rows: {total - bad_rows}")
print(f"Percentage corrupted: {round(bad_rows / total * 100, 3)}%")

Corrupted rows: 33
Clean rows: 50467
Percentage corrupted: 0.065%


33 corrupted rows out of 50,500 = 0.065% which is negligible. The fix here is simple and completely safe: we will drop those 33 rows. We lose nothing meaningful — our fraud signal (70 rows) is untouched, and our clean dataset still has 50,467 rows to work with.

Step 6 — Cleaning the data and fixing the amount type

In [0]:
from pyspark.sql.functions import col

# Fix 1: Dropping the 33 corrupted rows
# Keeping only rows where isFraud is exactly 0 or exactly 1
df_clean = df.filter(
    (col("isFraud") == 0.0) | (col("isFraud") == 1.0)
)

# Fix 2: Casting amount from string to decimal number
# .cast("double") converts the column type
df_clean = df_clean.withColumn(
    "amount", col("amount").cast("double")
)

# Fix 3: Casting isFraud to integer (0 or 1, not 0.0 or 1.0)
# This is cleaner for modeling later
df_clean = df_clean.withColumn(
    "isFraud", col("isFraud").cast("integer")
)

# Verifying the result
print("Clean row count:", df_clean.count())
print("Fraud cases remaining:", df_clean.filter(col("isFraud") == 1).count())
df_clean.printSchema()

Clean row count: 50467
Fraud cases remaining: 70
root
 |-- step: double (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: double (nullable = true)



Step 7 — Saving clean data to Delta Lake

In [0]:
from pyspark.sql.functions import col, expr

# Starting fresh from the original raw DataFrame
# Step 1: Keeping only rows where isFraud is exactly 0 or 1
df_clean = df.filter(
    (col("isFraud") == 0.0) | (col("isFraud") == 1.0)
)

In [0]:
# try_cast attempts the conversion to double
# if it finds something like 'About257762.86' it returns null
# instead of crashing like .cast() does
df_clean = df_clean.withColumn(
    "amount",
    expr("try_cast(amount as double)")
)

print("amount conversion done")

amount conversion done


In [0]:
# Counting how many rows got nulled out during the amount conversion
bad_amount_rows = df_clean.filter(col("amount").isNull()).count()
print(f"Rows with unconvertible amount: {bad_amount_rows}")

# Now drop those rows
df_clean = df_clean.filter(col("amount").isNotNull())
print(f"Rows remaining after drop: {df_clean.count()}")

Rows with unconvertible amount: 505
Rows remaining after drop: 49962


In [0]:
# Make sure we still have fraud cases after dropping bad amount rows
fraud_remaining = df_clean.filter(col("isFraud") == 1).count()
print(f"Fraud cases remaining: {fraud_remaining}")

Fraud cases remaining: 69


In [0]:
# Converting isFraud from 0.0/1.0 (decimal) to 0/1 (whole number)
# Models expect integer labels, not decimals
df_clean = df_clean.withColumn(
    "isFraud",
    col("isFraud").cast("integer")
)

print("isFraud cast done")

isFraud cast done


In [0]:
# Where we want to save the clean data
output_path = "dbfs:/Volumes/workspace/default/raw_data/clean"

# Write to Delta format
# mode("overwrite") replaces anything already at this path
df_clean.write.format("delta").mode("overwrite").save(output_path)

print("Clean data saved to Delta Lake successfully.")
print(f"Location: {output_path}")

Clean data saved to Delta Lake successfully.
Location: dbfs:/Volumes/workspace/default/raw_data/clean


Step 8 — Check for null values

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# For every column, count how many rows have a null value
null_counts = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
])

null_counts.show()

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



Step 9 — Transaction type distribution

In [0]:
from pyspark.sql.functions import mean

# For each transaction type, showing:
# 1. How many total transactions
# 2. How many were fraud
# 3. What percentage were fraud

df_clean.groupBy("type").agg(
    spark_sum(col("isFraud")).alias("fraud_count"),
    mean(col("isFraud")).alias("fraud_rate")
).orderBy("fraud_count", ascending=False).show()

+---------+-----------+--------------------+
|     type|fraud_count|          fraud_rate|
+---------+-----------+--------------------+
| TRANSFER|         35|0.008411439557798606|
| CASH_OUT|         34|0.001927219136152...|
|  CASH_IN|          0|                 0.0|
| PAYMENT |          0|                 0.0|
| CASH_IN |          0|                 0.0|
|    DEBIT|          0|                 0.0|
|  PAYMENT|          0|                 0.0|
|CASH_OUT |          0|                 0.0|
|TRANSFER |          0|                 0.0|
+---------+-----------+--------------------+

